# Remaining work on the ActionShap submission — run all to finish it

This notebook **does** the remaining work rather than listing it. Restart and run all: it verifies the
printed numbers, executes whatever cohort runs are still missing, regenerates and re-freezes the
release, rebuilds the documents, clears the one test marker that waits on a TeX build, and repacks the
submission archive. Every step reads the repository first and skips finished work, so it resumes
safely after an interruption and says so in its output.

The only things it will not do are the ones no machine can decide for you; §7 lists those.


In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path

def _root():                        # works from notebooks/, the project dir, or the repository root
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "code" / "scripts" / "make_remaining_work_notebook.py").exists():
            return base
    raise RuntimeError("run this notebook from inside the ActionShap project")

ROOT = _root()
CODE, PAPER = ROOT / "code", ROOT
REPO = next(a for a in [ROOT, *ROOT.parents] if (a / "Makefile").exists())   # the Makefile lives here
PY = sys.executable

APPLY = True        # False prints commands instead of running them (safe rehearsal)
SKIP_RUNS = False   # True does everything except the cohort queue
MAX_HOURS = None    # bound one sitting, e.g. 8.0; run all again to resume

def sh(argv, cwd=None, note=""):
    """Run argv, streaming nothing but returning everything; raise with the output on failure."""
    if not APPLY:
        print(f"[dry-run] cd {cwd or REPO} && {' '.join(map(str, argv))}")
        return "", 0
    started = time.time()
    proc = subprocess.run([str(a) for a in argv], cwd=str(cwd or REPO), capture_output=True, text=True)
    secs = time.time() - started
    tail = "\n".join((proc.stdout + "\n" + proc.stderr).strip().splitlines()[-8:])
    print(f"      {secs/60:6.1f} min  {(note or argv[-1])[:60]}")
    if proc.returncode != 0:
        print(tail)
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(map(str, argv))}")
    return proc.stdout, proc.returncode

def make(target, note=None):
    return sh(["make", target, f"PY={PY}"], cwd=REPO, note=note or f"make {target}")

sys.path.insert(0, str(PAPER / "code" / "scripts"))
import make_remaining_work_notebook as mrw

st = mrw.status()
print(f"repo {REPO}")
print(f"queue {st['n_jobs']} jobs ({st['done_jobs']} payloads already present, "
      f"~{st['queued_hours']:.0f} h left) | verified rows {st['audit_rows'] - st['audit_unsupported']}/"
      f"{st['audit_rows']} | stamp {st['manifest_stamp']} "
      f"{'ok' if st['stamp_matches'] else 'MISMATCH'} | stale PDFs {st['stale_pdfs'] or 'none'}")


## 1. Claims the queue cannot cover

Four items on the worklist are engineering or scope decisions, not compute. This is the only cell that touches the manuscript, and it is a single idempotent paragraph.

In [ ]:
# 1. Scope the claims the queue cannot cover. The four boundaries below are closed by stating them
#    exactly, so the paragraph must be typeset; this inserts it where it belongs if it is missing.
main = PAPER / "acmart-primary" / "acmmanuscript.tex"
text = main.read_text(encoding="utf-8")
if st["scope"]["all_present"]:
    print("scope paragraph already typeset: nothing to do")
elif not st["scope"]["anchor_present"]:
    raise RuntimeError("anchor sentence not found - the Limitations paragraph moved; re-locate it by hand")
else:
    new = text.replace(mrw.SCOPE_ANCHOR, mrw.SCOPE_ANCHOR + "\n\n" + mrw.SCOPE_PARAGRAPH, 1)
    for path in (main, PAPER / "actionshap-ipm" / "acmmanuscript.tex"):
        if path.exists() and mrw.SCOPE_ANCHOR in path.read_text(encoding="utf-8") \
           and not all(v in path.read_text(encoding="utf-8") for v in mrw.SCOPE_CLAUSES.values()):
            if APPLY:
                path.write_text(path.read_text(encoding="utf-8").replace(
                    mrw.SCOPE_ANCHOR, mrw.SCOPE_ANCHOR + "\n\n" + mrw.SCOPE_PARAGRAPH, 1))
                print(f"inserted scope paragraph into {path.relative_to(PAPER)}")
            else:
                print(f"[dry-run] would insert scope paragraph into {path.relative_to(PAPER)}")
after = mrw.status()["scope"]
print("clause check:", {k: v for k, v in after["clause_present"].items()})
assert after["all_present"], "the manuscript must state all four boundaries"


## 2. Verify the printed numbers

The supplement's decision-quality block is recomputed row by row from `user_seed_metrics.csv.gz`; success rates are the seed-averaged per-seed indicator, so values live on a 0.0002 lattice.

In [ ]:
# 2. Prove the numbers already in the paper, row by row, from the frozen matrices.
out, rc = sh([PY, str(PAPER / "code/scripts/audit_success_estimand.py"), "--check"], cwd=CODE,
             note="estimand audit")
aud = json.loads((CODE / "results/review9/success_estimand_audit.json").read_text())
print(f"rows {len(aud['rows'])}, unsupported {aud['unsupported']}, lattice "
      f"{aud['slice'].get('grid', 0):.6f} = 1/(n R_seed) -> every printed rate is reproducible")
print("(the audit is also wired into `make check`, so this cannot drift silently)")


## 3. The cohort queue

13 of 13 payloads still missing (~114.0 h). Set `SKIP_RUNS = True` in §0 to do everything else and leave these to another machine, or `MAX_HOURS` to bound this sitting.

In [ ]:
# 3. The cohort queue: only jobs whose payload is absent, only real subcommands, resumable.
jobs = [j for j in mrw.queue() if not j["done"]]
if SKIP_RUNS:
    print(f"SKIP_RUNS = True: {len(jobs)} jobs left ({sum(j['minutes'] for j in jobs)/60:.1f} h)")
elif not jobs:
    print("all queued payloads present: nothing to run")
else:
    total = sum(j["minutes"] for j in jobs) / 60.0
    print(f"running {len(jobs)} jobs (~{total:.1f} h plan); Ctrl-C is safe - finished payloads are kept\n")
    spent = 0.0
    for job in jobs:
        if MAX_HOURS is not None and spent >= MAX_HOURS:
            print(f"MAX_HOURS={MAX_HOURS:.1f} h reached after {spent:.1f} h; the remaining jobs "
                  f"resume on the next run all")
            break
        print(f"  {job['experiment']:<18} {job['dataset']:<10} users={job['users']}")
        sh([PY, "scripts/run_review9_experiments.py", *job["argv"]], cwd=CODE,
           note=f"{job['experiment']}/{job['dataset']}")
        spent += job["minutes"] / 60.0
    left = [j for j in mrw.queue() if not j["done"]]
    print(f"\npayloads now on disk: {st['n_jobs'] - len(left)}/{st['n_jobs']}"
          + ("" if not left else f"; still missing: {', '.join(j['experiment'] + '/' + j['dataset'] for j in left)}"))


## 4. Regenerate the release

Tables and statistics from the payloads, manifest re-frozen, stamp re-quoted in both documents, then the whole validator suite.

In [ ]:
# 4. Rebuild the release artifacts from the payloads, then make the documents quote the new stamp.
make("tables", "regenerate tables and statistics")
make("manifest", "re-freeze the manifest")
stamp = json.loads((CODE / "results/manifest.json").read_text())["manifest_stamp"]
for doc in ("acmart-primary/acmmanuscript.tex", "acmart-primary/supplementary.tex"):
    path = PAPER / doc
    body = path.read_text(encoding="utf-8")
    quoted = re.findall(r"\\newcommand\{\\resultmanifeststamp\}\{([0-9a-f]+)\}", body)
    if quoted == [stamp]:
        print(f"      stamp already {stamp} in {doc}")
    else:
        new = re.sub(r"(\\newcommand\{\\resultmanifeststamp\})\{[0-9a-f]*\}",
                     lambda m: m.group(1) + "{" + stamp + "}", body, count=1)
        if APPLY:
            path.write_text(new)
        print(f"      re-quoted {doc}: {quoted} -> {stamp}")
make("check", "validators + suite (before the PDF build)")
print(f"release rebuilt at {stamp} ({len(json.loads((CODE / 'results/manifest.json').read_text())['files'])} files)")


## 5. Build the documents

In [ ]:
# 5. Documents. Needs a TeX engine; if none is installed the notebook says so instead of faking it,
#    and the archive in §6 still rebuilds so an Overleaf compile is a valid substitute.
engine = next((e for e in ("latexmk", "pdflatex", "tectonic") if shutil.which(e)), None)
built = False
if engine is None:
    print("no TeX engine on PATH: `make pdf` skipped.")
    print("    either  brew install --cask mactex-no-gp  then run all again,")
    print("    or compile the archive in §6 on Overleaf and drop the two PDFs back into acmart-primary/.")
else:
    print(f"building with {engine}")
    make("pdf", "build both documents")
    built = True

# The suite carries one xfail marker that exists only because the committed PDFs predate the text fixes.
# Clear it *only* if the rebuilt PDFs actually satisfy the test; otherwise restore it.
tests = CODE / "tests/test_review9_publication_integrity.py"
marker = re.compile(r"[ \t]*@pytest\.mark\.xfail[^\n]*\n")
src = tests.read_text(encoding="utf-8")
if not built:
    print("PDF-freshness marker left in place (documents not rebuilt)")
elif APPLY and marker.search(src):
    tests.write_text(marker.sub("", src, count=1))
    probe = sh([PY, "-m", "pytest", "tests/test_review9_publication_integrity.py", "-q"], cwd=CODE,
               note="suite without the marker")
    if "failed" in probe[0]:
        tests.write_text(src)
        print("test still fails -> marker restored; inspect the PDF build")
    else:
        print("marker removed: the rebuilt PDFs carry the revised text")
else:
    print("marker already cleared" if not marker.search(src) else "dry-run: would clear the marker")


## 6. Package and gate

In [ ]:
# 6. Submission archive and the final gate.
make("overleaf", "repack the Overleaf archive")
zip_path = PAPER / "actionshap-overleaf.zip"
if zip_path.exists():
    import zipfile
    names = zipfile.ZipFile(zip_path).namelist()
    tex = zipfile.ZipFile(zip_path).read("acmmanuscript.tex").decode()
    quoted = re.search(r"resultmanifeststamp\}\{([0-9a-f]+)", tex)
    print(f"      {len(names)} files; archive quotes stamp "
          + (quoted.group(1) if quoted else "MISSING"))

st2 = mrw.status()
done = {
    "printed rows reproducible from the release": st2["audit_unsupported"] == 0,
    "every queued payload present": not st2["pending_jobs"],
    "documents quote the current manifest stamp": st2["stamp_matches"],
    "boundaries stated in the text": st2["scope"]["all_present"],
    "tables regenerated and validators green": True,      # §4 raises otherwise
}
open_items = []
if st2["stale_pdfs"]:
    open_items.append("PDFs not rebuilt here (no TeX engine) - compile in Overleaf or install one")
open_items.append("upload the archive; register the artifact DOI/license after acceptance")
for name, ok in done.items():
    print(f"  [{'x' if ok else ' '}] {name}")
print("\nCLOSED (everything a machine can decide is done)" if all(done.values())
      else "\nNOT CLOSED: " + ", ".join(k for k, v in done.items() if not v))
print("left for you:\n  - " + "\n  - ".join(open_items))
print(f"\ncommit with:  git add -A && git commit -m 'regenerated after the run queue' && git push")


## 7. What deliberately stays with the authors

* **Venue formatting.** The documents are ACM-formatted. Converting to another journal's template is a formatting task with no bearing on the analysis, and doing it blind would risk the tables.
* **Deposit registration.** A DOI, a license choice and the public URL are actions on an external service; the archive references the frozen manifest, which is reproducible without them.
* **Claim widening.** If you implement a competitive-model attribution audit or adaptive stopping, delete the corresponding sentence in §1's paragraph and re-run; the text then asserts what the payload supports. Nothing in this notebook widens a claim on its own.